# Chapter 3 - Lab 2: <font color='blue'>Google ADK Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using Google's Agent Development Kit (ADK)** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

Google ADK introduces three building blocks that recur across enterprise frameworks:

* An **LlmAgent** that wraps the model, system instruction, and tools,
* A **SessionService** that holds conversation history and state,
* A **Runner** that orchestrates the agent loop as an asynchronous stream of events.

Notice that ADK is **async-first** — unlike LangChain's `agent.invoke(...)`, you stream events with `async for`.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **Google ADK** (`google-adk`) — agent / session / runner abstraction.
* **Gemini 2.5 Flash** — the underlying LLM (via `google-genai`).
* **Python `asyncio`** — required to drive the event stream.

You will need a Google AI Studio API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q google-adk google-genai pydantic python-dotenv

## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`GOOGLE_API_KEY`** — your Google AI Studio key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Create the ADK agent, session, and runner

ADK separates concerns: the `LlmAgent` describes *what* the agent is, the `SessionService` keeps the conversation, and the `Runner` actually executes it. We use `InMemorySessionService` for this lab — production deployments would swap in a persistent backend.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME, USER_ID, SESSION_ID = 'finance_app', 'user_1', 'session_001'

finance_agent = LlmAgent(
    model='gemini-2.5-flash',
    name='finance_agent',
    instruction=(
        'Use get_stock_data for AAPL and JPM, then compute their P/E with compute_pe. '
        'Return a short memo with both prices and P/E ratios.'
    ),
    tools=[get_stock_data, compute_pe],
)

Now wire up the session service and the runner. Both are async — note `await` on `create_session`.

In [ ]:
session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID,
)
runner = Runner(
    agent=finance_agent,
    app_name=APP_NAME,
    session_service=session_service,
)
print(f'Session created: {SESSION_ID}')

## 5. Run the agent and stream events

We call `runner.run_async` and iterate through the event stream. ADK exposes **every step** the agent takes — tool calls, intermediate thoughts, and the final response — as discrete events. Here we wait for the final response event.

In [ ]:
content = types.Content(role='user', parts=[types.Part(text=input_message)])

final_response_text = 'Agent did not produce a final response.'
async for event in runner.run_async(
    user_id=USER_ID, session_id=SESSION_ID, new_message=content,
):
    if event.is_final_response():
        if event.content and event.content.parts:
            final_response_text = event.content.parts[0].text
        break

print(final_response_text)

## 6. Inspect intermediate events (optional)

If you want to see every step the agent took, iterate through events again and print them. This is the trace ADK gives you for free — invaluable for debugging non-trivial agent flows in production.

In [ ]:
async for event in runner.run_async(
    user_id=USER_ID, session_id=SESSION_ID, new_message=content,
):
    print(f'[{type(event).__name__}] is_final={event.is_final_response()}')

## 7. Results

You should see the agent call `get_stock_data` (once per ticker), then call `compute_pe` to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about Google ADK specifically:**

* ADK is **async-first** — you stream events rather than receive a single return value.
* The clear separation between `LlmAgent`, `SessionService`, and `Runner` mirrors enterprise patterns where conversation state and execution are managed by different services.
* Out of the box you get a per-event trace, making observability easier than in frameworks that hide the loop.
* Trade-off: more code than LangChain's `create_agent` for the same simple task — the abstractions pay off as complexity grows.